In [7]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
!mkdir -p "/content/drive/MyDrive/Iraq_Lake/images"
!mkdir -p "/content/drive/MyDrive/Iraq_Lake/masks"
!mv "/content/drive/MyDrive/Iraq_Lake/"*"_s2_4band"*.tif* "/content/drive/MyDrive/Iraq_Lake/images/" 2>/dev/null || true
!mv "/content/drive/MyDrive/Iraq_Lake/"*"_gsw_water"*.tif* "/content/drive/MyDrive/Iraq_Lake/masks/" 2>/dev/null || true

!echo "IMAGES:"; ls "/content/drive/MyDrive/Iraq_Lake/images" | head -n 20
!echo "MASKS:";  ls "/content/drive/MyDrive/Iraq_Lake/masks"  | head -n 20

image_dir = "/content/drive/MyDrive/Iraq_Lake/images"
mask_dir  = "/content/drive/MyDrive/Iraq_Lake/masks"

IMAGES:
darbandikhan_2017_s2_4band.tif
darbandikhan_2021_s2_4band.tif
dukan_2017_s2_4band.tif
dukan_2021_s2_4band.tif
habbaniyah_2017_s2_4band.tif
habbaniyah_2021_s2_4band.tif
razzaza_2017_s2_4band.tif
razzaza_2021_s2_4band.tif
tharthar_2017_s2_4band.tif
tharthar_2021_s2_4band.tif
MASKS:
darbandikhan_2017_gsw_water.tif
darbandikhan_2021_gsw_water.tif
dukan_2017_gsw_water.tif
dukan_2021_gsw_water.tif
habbaniyah_2017_gsw_water.tif
habbaniyah_2021_gsw_water.tif
razzaza_2017_gsw_water.tif
razzaza_2021_gsw_water.tif
tharthar_2017_gsw_water.tif
tharthar_2021_gsw_water.tif


In [9]:
# ====== Iraq Lake dataset: tile + (Amazon-like) preprocessing + save .npy ======
# Outputs:
# /content/drive/MyDrive/Iraq_Lake/iraq-processed-large/
#   training/images/*.npy, training/masks/*.npy
#   validation/images/*.npy, validation/masks/*.npy
#   test/images/*.npy, test/masks/*.npy

import os, glob, random
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# ---------------------------
# 0) PATHS + SETTINGS
# ---------------------------
GEE_IMG_DIR = "/content/drive/MyDrive/Iraq_Lake/images"
GEE_MSK_DIR = "/content/drive/MyDrive/Iraq_Lake/masks"

OUT_ROOT = "/content/drive/MyDrive/Iraq_Lake/iraq-processed-large"

TILE = 512
SEED = 42
TRAIN, VAL, TEST = 0.68, 0.27, 0.05

# If True: skip tiles where mask is all zeros (no water)
SKIP_EMPTY_MASK_TILES = False

random.seed(SEED)
np.random.seed(SEED)

# ---------------------------
# 1) HELPERS
# ---------------------------
def make_dirs(root: str) -> None:
    for split in ["training", "validation", "test"]:
        os.makedirs(os.path.join(root, split, "images"), exist_ok=True)
        os.makedirs(os.path.join(root, split, "masks"), exist_ok=True)

def align_mask_to_image(img_path: str, mask_path: str):
    """
    Returns:
      img_arr: (H, W, 4) float32
      mask_arr: (H, W) uint8 in {0,1}
    """
    with rasterio.open(img_path) as src_img:
        img = src_img.read().astype(np.float32)   # (bands,H,W)
        img_transform = src_img.transform
        img_crs = src_img.crs
        img_h, img_w = src_img.height, src_img.width

    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

    with rasterio.open(mask_path) as src_msk:
        msk = src_msk.read(1).astype(np.float32)
        msk = np.nan_to_num(msk, nan=0.0, posinf=0.0, neginf=0.0)

        same_grid = (
            src_msk.crs == img_crs and
            src_msk.transform == img_transform and
            src_msk.width == img_w and
            src_msk.height == img_h
        )

        if same_grid:
            mask_resampled = msk
        else:
            mask_resampled = np.zeros((img_h, img_w), dtype=np.float32)
            reproject(
                source=msk,
                destination=mask_resampled,
                src_transform=src_msk.transform,
                src_crs=src_msk.crs,
                dst_transform=img_transform,
                dst_crs=img_crs,
                resampling=Resampling.nearest
            )

    img_arr = np.transpose(img, (1, 2, 0))             # (H,W,4)
    mask_arr = (mask_resampled > 0.5).astype(np.uint8) # (H,W) 0/1
    return img_arr, mask_arr

def per_tile_minmax(x: np.ndarray) -> np.ndarray:
    """
    Mimics their preprocessing:
      (a - min(a)) / (max(a) - min(a))
    Applied PER TILE (not dataset-wide).
    """
    x = x.astype(np.float32)
    mn = float(np.min(x))
    mx = float(np.max(x))
    if mx <= mn:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn)

def collect_tiles():
    """
    Reads each lake-year scene:
      *_s2_4band.tif  +  *_gsw_water.tif
    Aligns mask to image (if needed),
    then tiles into 512x512 patches.
    Returns a list of dicts: {id, img, msk}
    """
    s2_files = sorted(glob.glob(os.path.join(GEE_IMG_DIR, "*_s2_4band.tif*")))
    if not s2_files:
        raise FileNotFoundError(f"No S2 files found in: {GEE_IMG_DIR}")

    tiles = []
    for s2_path in s2_files:
        fname = os.path.basename(s2_path)
        base = fname.replace("_s2_4band.tif", "").replace("_s2_4band.tiff", "")

        # mask filename is base + _gsw_water.tif
        msk_path = os.path.join(GEE_MSK_DIR, base + "_gsw_water.tif")
        if not os.path.exists(msk_path):
            msk_path = os.path.join(GEE_MSK_DIR, base + "_gsw_water.tiff")

        if not os.path.exists(msk_path):
            print(f"⚠ Missing mask for {base}, skipping")
            continue

        print("Reading + aligning:", base)
        img_arr, mask_arr = align_mask_to_image(s2_path, msk_path)

        H, W, C = img_arr.shape
        if C != 4:
            print(f"⚠ Expected 4 bands, got {C} for {base}. Continuing anyway.")

        # tile without padding (only full 512 tiles)
        for r0 in range(0, H - TILE + 1, TILE):
            for c0 in range(0, W - TILE + 1, TILE):
                img_t = img_arr[r0:r0+TILE, c0:c0+TILE, :]
                msk_t = mask_arr[r0:r0+TILE, c0:c0+TILE]

                if SKIP_EMPTY_MASK_TILES and int(msk_t.sum()) == 0:
                    continue

                tile_id = f"{base}_r{r0//TILE:03d}_c{c0//TILE:03d}"
                tiles.append({"id": tile_id, "img": img_t, "msk": msk_t})

    if not tiles:
        raise RuntimeError("No tiles created. Check your raster sizes and paths.")
    return tiles

def split_tiles(tiles):
    """
    RANDOM split at tile level (fast).
    NOTE: can leak if adjacent tiles from same scene end up in different splits.
    """
    random.shuffle(tiles)
    n = len(tiles)
    n_train = int(round(TRAIN * n))
    n_val = int(round(VAL * n))
    n_train = min(n_train, n)
    n_val = min(n_val, n - n_train)

    train_tiles = tiles[:n_train]
    val_tiles   = tiles[n_train:n_train+n_val]
    test_tiles  = tiles[n_train+n_val:]
    return train_tiles, val_tiles, test_tiles

def save_as_npy(tiles_list, split_name):
    """
    Saves:
      image: (512,512,4) float32 after per-tile min-max norm
      mask:  (512,512,1) uint8 (0/1)
    """
    img_out = os.path.join(OUT_ROOT, split_name, "images")
    msk_out = os.path.join(OUT_ROOT, split_name, "masks")

    for t in tiles_list:
        x = per_tile_minmax(t["img"]).astype(np.float32)  # (512,512,4)
        y = t["msk"].astype(np.uint8)[..., None]          # (512,512,1)

        np.save(os.path.join(img_out, t["id"] + ".npy"), x)
        np.save(os.path.join(msk_out, t["id"] + ".npy"), y)

# ---------------------------
# 2) RUN
# ---------------------------
make_dirs(OUT_ROOT)

tiles = collect_tiles()
print("Total tiles:", len(tiles))

train_tiles, val_tiles, test_tiles = split_tiles(tiles)
print("Train/Val/Test:", len(train_tiles), len(val_tiles), len(test_tiles))

save_as_npy(train_tiles, "training")
save_as_npy(val_tiles, "validation")
save_as_npy(test_tiles, "test")

print("✅ Done. Saved to:", OUT_ROOT)



Reading + aligning: darbandikhan_2017
Reading + aligning: darbandikhan_2021
Reading + aligning: dukan_2017
Reading + aligning: dukan_2021
Reading + aligning: habbaniyah_2017
Reading + aligning: habbaniyah_2021
Reading + aligning: razzaza_2017
Reading + aligning: razzaza_2021
Reading + aligning: tharthar_2017
Reading + aligning: tharthar_2021
Total tiles: 1260
Train/Val/Test: 857 340 63
✅ Done. Saved to: /content/drive/MyDrive/Iraq_Lake/iraq-processed-large
